In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("เชื่อมต่อ Drive สำเร็จ!")

In [ ]:
!pip install -q pycocotools mediapipe opencv-python-headless tqdm
import cv2, mediapipe as mp, numpy as np
print("ติดตั้งเครื่องมือ MediaPipe และ OpenCV เรียบร้อย!")

In [ ]:
import os

# กำหนด Path หลักที่เคยสร้างไว้ใน Notebook แรก
BASE = '/content/drive/MyDrive/exercise_pose'
ANNO = f'{BASE}/data/coco/annotations'
IMGS = f'{BASE}/data/coco/images'
PROC = f'{BASE}/data/processed'

print(f"ระบบพร้อมทำงานที่โฟลเดอร์: {BASE}")

In [ ]:
import pickle
import numpy as np
import os

# ใช้ Path ที่เรา Copy มาเมื่อวาน (ตรวจสอบให้แน่ใจว่ารัน Setup Cell ก่อนนะครับ)
FILE_PATH = f'{PROC}/samples_val_filtered.pkl'

with open(FILE_PATH, 'rb') as f:
    samples = pickle.load(f)

print(f"✅ โหลดข้อมูลสำเร็จ! พบข้อมูลคุณภาพสูง: {len(samples):,} รายการ")

In [ ]:
def calc_angle(a, b, c):
    """
    คำนวณมุมที่จุด b (หน่วย: องศา) จากพิกัด 3 จุด
    a, b, c: พิกัด [x, y] ของแต่ละจุด
    """
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)

    # สร้างเวกเตอร์จากจุด b ไปยัง a และ c
    ba = a - b
    bc = c - b

    # คำนวณหา Cosine ของมุมระหว่างเวกเตอร์ด้วย Dot Product
    cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-7)
    angle = np.arccos(np.clip(cosine_angle, -1.0, 1.0))

    return np.degrees(angle)

# ทดลองคำนวณมุม 90 องศา (สมมติจุด)
print(f"ทดสอบมุมฉาก: {calc_angle([0, 1], [0, 0], [1, 0]):.1f} องศา")

In [ ]:
# ดึงข้อมูลคนแรก
person = samples[0]
kps = person['kps'] # พิกัด (17, 3) -> [x, y, v]

# ดึงเฉพาะค่า X, Y ของ ขาข้างซ้าย
left_hip = kps[11][:2]
left_knee = kps[13][:2]
left_ankle = kps[15][:2]

# คำนวณมุมเข่าซ้าย
knee_angle = calc_angle(left_hip, left_knee, left_ankle)

print(f"คนแรกในข้อมูล: Image ID {person['img_id']}")
print(f"มุมเข่าซ้ายปัจจุบัน: {knee_angle:.2f} องศา")

In [ ]:
def extract_features(kps):
    # kps: np.array shape (17, 3) = [x, y, visibility]
    # return: dict vao features

    f = {}

    # มุมเข่าซ้าย: hip(11) - knee(13) - ankle(15)
    if all(kps[i, 2] > 0 for i in [11, 13, 15]):
        f['left_knee'] = calc_angle(
            kps[11,:2], kps[13,:2], kps[15,:2])

    # มุมเข่าขวา: hip(12) - knee(14) - ankle(16)
    if all(kps[i, 2] > 0 for i in [12, 14, 16]):
        f['right_knee'] = calc_angle(
            kps[12,:2], kps[14,:2], kps[16,:2])

    # มุมสะโพกซ้าย: shoulder(5) - hip(11) - knee(13)
    if all(kps[i, 2] > 0 for i in [5, 11, 13]):
        f['left_hip'] = calc_angle(
            kps[5,:2], kps[11,:2], kps[13,:2])

    # มุมสะโพกขวา: shoulder(6) - hip(12) - knee(14)
    if all(kps[i, 2] > 0 for i in [6, 12, 14]):
        f['right_hip'] = calc_angle(
            kps[6,:2], kps[12,:2], kps[14,:2])

    # ความสมมาตร ซ้าย-ขวา
    if 'left_knee' in f and 'right_knee' in f:
        f['knee_symmetry'] = abs(f['left_knee'] - f['right_knee'])

    return f

In [ ]:
# 1. ดึงพิกัด (kps) ของคนลำดับแรกสุดใน Dataset มาเป็นหนูทดลอง
person_kps = samples[0]['kps']

# 2. โยนพิกัดเข้าไปใน "เครื่องปั่น" (ฟังก์ชัน) ที่เราเพิ่งสร้าง
features_result = extract_features(person_kps)

# 3. พิมพ์ผลลัพธ์ที่ได้ออกมาดู
print("🎯 สกัดฟีเจอร์สำเร็จ! นี่คือองศาข้อต่อที่เครื่องคำนวณได้:")
for joint_name, angle_value in features_result.items():
    print(f"- {joint_name}: {angle_value:.1f} องศา")

In [ ]:
import pandas as pd
from tqdm.notebook import tqdm

rows = []
for s in tqdm(samples, desc="Extracting features"):
    feat = extract_features(s['kps'])
    if feat:
        feat['img_id']= s['img_id']
        feat['area'] = s['area']
        rows.append(feat)

df = pd.DataFrame(rows)
print(df.shape)
print(df.describe().round(1))

# เซฟเป็น CSV และ pkl
df.to_csv(f'{PROC}/features_val.csv', index=False)
df.to_pickle(f'{PROC}/features_val.pkl')
print(f"เซฟแล้ว: {len(df):,} rows, {df.shape[1]} features")

In [ ]:
import os
os.chdir('/content/exercise-pose')

# เช็คไฟล์ที่อยู่ใน repo ตอนนี้
!git ls-files

In [ ]:
!git reset --soft HEAD~1

In [ ]:
# --- ส่วนนี้คือส่วนจัดการ GitHub ที่แก้ไขแล้ว ---
from google.colab import userdata
import os

# 1. ตั้งค่าพื้นฐาน (กรอกข้อมูลของคุณที่นี่)
GIT_TOKEN = userdata.get('GITHUB_TOKEN')
GIT_USERNAME = "wanasnanwangmad-cmd"
GIT_REPO = "exercise-pose"
REPO_URL = f"https://{GIT_TOKEN}@github.com/{GIT_USERNAME}/{GIT_REPO}.git"
REPO_DIR = f'/content/{GIT_REPO}'

# 2. ตั้งค่าตัวตน (Email/Name)
!git config --global user.email "wanasnan.wangmad@gmail.com"
!git config --global user.name "Wanasnan Wangmad"

# 3. เตรียม Folder (ถ้าไม่มีให้ Clone ถ้ามีให้ดึงข้อมูลล่าสุด)
if not os.path.exists(REPO_DIR):
    %cd /content
    !git clone {REPO_URL}
else:
    %cd {REPO_DIR}
    !git pull {REPO_URL} main

# 4. คัดลอกไฟล์จาก Drive เข้าสู่ Repo Folder
# ตรวจสอบ Path Drive ของคุณ (จากภาพที่คุณส่งมาคือ /content/drive/MyDrive/Colab Notebooks)
DRIVE_NB_DIR = '/content/drive/MyDrive/Colab Notebooks'
target_files = ['01_setup_and_data.ipynb', '02_explore.ipynb', '03_pipeline.ipynb', '04_features.ipynb']

for nb in target_files:
    src = f"{DRIVE_NB_DIR}/{nb}"
    dst = f"{REPO_DIR}/{nb}"
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"✅ คัดลอกสำเร็จ: {nb}")
    else:
        print(f"⚠️ ไม่พบไฟล์ {nb} ใน Drive (ลองเช็ค Path อีกครั้ง)")

# 5. สั่ง Push งาน
%cd {REPO_DIR}
!git add .
# กันเหนียว: ถ้าไม่มีอะไรเปลี่ยน commit จะไม่ทำงานและไม่พ่น Error
!git diff-index --quiet HEAD || git commit -m "update: cleaned notebooks and fixed pose logic"
!git push {REPO_URL} main

print("🚀 เรียบร้อย! โค้ดถูกส่งขึ้น GitHub แล้ว")